# 歌词采集-QQ音乐
不需要按专辑采集

In [1]:
import requests
import re
import json
import os
import time
import pandas as pd
import html
from datetime import datetime
from collections import defaultdict


from collections import Counter


In [2]:
import sys

sys.path.append('..')

from data_crawler import format_timestamp, search_song, get_songs_data_raw

# 李宇春

In [3]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

file_path_prefix = "data/liyuchun/"
singer = "李宇春"
max_page = 14

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [ ]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [ ]:
df_song_data_raw_read

### 清洗

In [ ]:
def clear_song_name(df):
    df = df.copy()
    # 歌曲数据
    df['song_name_unique'] = df['song_name'].astype(str)
    # 清洗song_name_unique，删除空格，将中文括号转为英文括号
    df['song_name_unique'] = df['song_name_unique'].str.replace(' ', '')
    df['song_name_unique'] = df['song_name_unique'].str.replace(
        '（', '(').str.replace('）', ')')
    # 删除括号内的内容
    df['song_name_pure'] = df['song_name_unique'].str.replace(r'\(.*?\)',
                                                              '',
                                                              regex=True)
    # 删除空格
    df['song_name_pure'] = df['song_name_pure'].str.replace(' ', '')

    return df

In [ ]:
# 选择歌手独唱作品
def clear_song_singer(df, singer_name):
    df = df.copy()
    df = df[df['artist_name'] == singer_name]
    return df

In [ ]:
# 删除疑似翻唱的作品，按专辑名筛选
def clear_song_tv_show(df, albums):
    df = df.copy()
    df['album_name_pure'] = df['album_name'].astype(str).fillna('无')
    for album in albums:
        df = df[~df['album_name_pure'].str.startswith(album)]
    return df

In [ ]:
albums_to_delete = ['声生不息', '在吗']

In [ ]:
df_songs = clear_song_name(df_song_data_raw_read)
df_songs = clear_song_singer(df_songs, singer)
df_songs = clear_song_tv_show(df_songs, albums_to_delete)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)
df_songs

In [ ]:
# 前130首
songs_list_all = df_songs['song_name_pure'].to_list()
songs_list_130 = df_songs.head(130)['song_name_pure'].to_list()


## 歌单确认

In [ ]:
songs_to_add = [
    '皇后与梦想', '下雨', '冰菊物语', '我的王国', '漂浮地铁', '今天有朵云爱我', '您所拨打的电话号码是空号',
    '一而再再而三地喜欢你', '人间乐园', 'TMD我爱你', '口音', '木兰', '开放'
]
songs_to_delete = ['今夜你会不会来', '春风十里', '情书', '那女孩对我说', '南方姑娘', '爱你所爱', '无心睡眠', '莫过于此', '天黑黑', '张三的歌', '漂洋过海来看你', '城里的月光', '不要对他说', '下个,路口,见']

In [ ]:
# 添加
songs_list_final = songs_list_130.copy()
for i in songs_to_add:
    if i not in songs_list_final:
        print(i)
        songs_list_final.append(i)

In [ ]:
# 删除
for i in songs_to_delete:
    if i in songs_list_final:
        print(i)
        songs_list_final.remove(i)

In [ ]:
len(songs_list_final)

## 歌曲数据确认

In [ ]:
df_songs_final = df_songs[df_songs['song_name_pure'].isin(
    songs_list_final)].reset_index(drop=True)

df_songs_final = df_songs_final.drop(columns=['song_name_unique'])
df_songs_final['song_name_unique'] = df_songs_final['song_name_pure']
df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
    lambda x: format_timestamp(x))
df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
    lambda x: x.split('-')[0])
df_songs_final

In [ ]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

## 歌词采集

In [ ]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)

## 歌词清洗

In [ ]:
clear_and_save_lyric(file_path_prefix, df_songs_final)